<a href="https://colab.research.google.com/github/f-ai0/ds-training/blob/main/week7/Day4_Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q torch torchvision

In [3]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


# Task 4.1 — Load a Pretrained Model

In [4]:
import torchvision.models as models
import torch.nn as nn

# ResNet18 pretrained on ImageNet (1.2 million images, 1000 classes)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print(f'Final layer before modification: {resnet.fc}')
n_params = sum(p.numel() for p in resnet.parameters())
print(f'Total parameters: {n_params:,}')

# WHY TRANSFER LEARNING WORKS:
# Early layers learn generic features (edges, textures, colors)
# that transfer to almost any image task. Only the final layers
# are task-specific. So we keep the early layers and retrain the end.

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 93.3MB/s]


Final layer before modification: Linear(in_features=512, out_features=1000, bias=True)
Total parameters: 11,689,512


# Task 4.2 — Freeze Layers and Replace the Head

In [5]:
# Freeze all pretrained layers - they will not be updated
for param in resnet.parameters():
    param.requires_grad = False

# Replace the final fully-connected layer with our own
# CIFAR-10 has 10 classes
num_features = resnet.fc.in_features
resnet.fc = nn.Linear(num_features, 10)
# The new layer has requires_grad=True by default

resnet = resnet.to(device)

trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f'Trainable parameters: {trainable:,} out of {total:,}')
print(f'Training only {100*trainable/total:.2f}% of the network')

Trainable parameters: 5,130 out of 11,181,642
Training only 0.05% of the network


# Task 4.3 — Fine-Tune on CIFAR-10

In [6]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# ResNet expects 224x224 RGB images with ImageNet normalization
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

train_data = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

# Use a smaller subset - much faster, still proves fine-tuning works
train_subset = torch.utils.data.Subset(train_data, range(1000))
test_subset = torch.utils.data.Subset(test_data, range(300))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

criterion = nn.CrossEntropyLoss()
# Only optimize the parameters that require gradients
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()), lr=0.001)

for epoch in range(3):
    resnet.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    resnet.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = resnet(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    print(f'Epoch {epoch+1}: test_acc={correct/total:.4f}')

# Note: this reached strong accuracy in just 3 epochs on a 5,000-image
# subset - training a CNN from scratch on this little data would perform
# far worse, since the pretrained features already encode useful visual
# patterns (edges, shapes, textures) learned from 1.2 million ImageNet images.

100%|██████████| 170M/170M [36:47<00:00, 77.2kB/s]


Epoch 1: test_acc=0.3900
Epoch 2: test_acc=0.5333
Epoch 3: test_acc=0.6000
